# WeakTr: final 11 epochs and final evaluation

Use this notebook **after the preceding run has produced a checkpoint with 89 completed Phase-2 epochs**. It verifies the restored snapshot, trains the remaining epochs through 100, runs official multiscale evaluation and CRF, writes final scores/reports, and exports the final resume ZIP. Phase 1, CAM generation, and pseudo-label generation are skipped.

Attach only these two private datasets:

1. Original VOC/ImageNet inputs (`weaktr-inputs-manifest.json`).
2. The **newest extracted `weaktr-resume.zip` from the 89-epoch run**, including the actual Phase-1/Phase-2 snapshots, `phase2/variant.yml`, complete `pseudo/`, manifests, logs, and records. An active artifact folder from the same run also works. Remove older saved-run datasets.

Start a fresh Kaggle session, enable **Internet and GPU T4**, and run all cells. Defaults are already set: `START_FROM='phase2'`, `PHASE2_PILOT=False`, `PHASE2_EPOCHS_THIS_SESSION=11`, and `MIN_PHASE2_COMPLETED_EPOCHS=89`. An older checkpoint causes a clear stop instead of silently running an incomplete final session. If the restored snapshot is already at 100 epochs, use `START_FROM='evaluate'` to repeat only final evaluation.

The last 11 training/validation epochs are estimated at **3.7 hours**, plus setup and final evaluation/CRF. Final evaluation runtime has not been measured on this run, so finishing before 12 hours is not guaranteed. Training exports its checkpoint ZIP before final evaluation; if evaluation fails or times out, recover that 100-epoch run and resume with `START_FROM='evaluate'`.

Environment/data/CAM/probability scratch stays in `/kaggle/tmp/weaktr-runtime`. Durable results, final CRF masks and `/kaggle/working/weaktr-resume.zip` stay in working outputs. No Windows-side helper files are needed. Completion does not guarantee the paper's 74.0% target.

In [ ]:
# 1. Final-session configuration: restore 89 completed epochs and finish the experiment run.
from pathlib import Path
import json, os, subprocess, sys

if not Path('/kaggle').is_dir():
    raise RuntimeError('Run in Kaggle or select the Kaggle remote Jupyter kernel.')

REPO = Path('/kaggle/working/WeakTr')
WORK = Path('/kaggle/working')
INPUTS = Path('/kaggle/input')
SCRATCH = Path('/kaggle/tmp/weaktr-runtime')
RESUME_FROM = 'auto'  # Or the exact directory containing the desired run-manifest.json.
START_FROM = 'phase2'  # phase1 | cams | pseudo | phase2 | evaluate
RUN_SMOKE = False   # Optional: recheck both models in this GPU session.
PHASE2_PILOT = False  # True: one Phase-2 epoch, then stop before full training/evaluation.
PHASE2_EPOCHS_THIS_SESSION = 11  # Finish 89 -> 100; the official total remains 100.
MIN_PHASE2_COMPLETED_EPOCHS = 89  # Refuse an older saved run for this final-session notebook.

STAGES = ('phase1', 'cams', 'pseudo', 'phase2', 'evaluate')
if START_FROM not in STAGES:
    raise ValueError(f'START_FROM must be one of {STAGES}')

if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', 'reproduction/voc-deit-small',
                    'https://github.com/ntmphu/WeakTr.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin',
                    'reproduction/voc-deit-small'], check=True)

def workflow(stage, *, skip_install=False):
    command = [sys.executable, str(REPO / 'reproduction/kaggle.py'), stage,
               '--inputs', str(INPUTS), '--work', str(WORK), '--scratch', str(SCRATCH),
               '--resume-from', RESUME_FROM]
    if stage == 'phase2':
        command += ['--epochs-this-session', str(PHASE2_EPOCHS_THIS_SESSION)]
    if skip_install:
        command.append('--skip-install')
    subprocess.run(command, cwd=REPO, check=True)

def should_run(stage):
    return STAGES.index(stage) >= STAGES.index(START_FROM)

print({'kernel': sys.executable, 'start_from': START_FROM, 'resume': RESUME_FROM,
       'temporary': str(SCRATCH), 'saved_outputs': str(WORK)})

def phase2_complete():
    script = ('import sys; from pathlib import Path; '
              'from reproduction.checkpoint import load_checkpoint; '
              'p = Path(sys.argv[1]); s = load_checkpoint(p) if p.is_file() else {}; '
              'assert not s or (s.get("stage") == "phase2" and s.get("format_version") == 1); '
              'print(s.get("epoch", -1) + 1)')
    result = subprocess.run([str(SCRATCH / 'weaktr-env/bin/python'), '-c', script,
                             str(WORK / 'weaktr-artifacts/phase2/checkpoint.pth')],
                            cwd=REPO, stdout=subprocess.PIPE, text=True, check=True)
    completed = int(result.stdout.strip())
    print(f'Phase 2 checkpoint: {completed}/100 completed epochs')
    if completed < MIN_PHASE2_COMPLETED_EPOCHS:
        raise RuntimeError(f'This final session requires at least {MIN_PHASE2_COMPLETED_EPOCHS} completed epochs; restored {completed}. Attach the newest 89-epoch saved run.')
    return completed >= 100


## 2. Restore and prepare

Restores an attached saved run with hash verification and validates inputs. Setup hides CUDA and does not train. Existing local checkpoints take precedence in `auto` mode. A complete installed environment is reused automatically; missing or broken dependencies trigger normal installation. Both trainer/CRF imports, CPU tests, and real-data checks still run.

Wait for **PREPARATION PASSED**. If the old session ran out of space, an optional cell `!python -m pip cache purge` clears installer downloads without deleting checkpoints. The launcher relocates known old temporary paths and reports storage usage/free capacity. Scratch relocation reduces saved output size; it cannot create physical capacity on a shared full filesystem.

In [ ]:
# Prefer the accessible active Phase-2 folder over an older exported ZIP.
if START_FROM in ('phase2', 'evaluate') and RESUME_FROM == 'auto' and not (WORK / 'weaktr-artifacts/phase2/checkpoint.pth').is_file():
    candidates = sorted({p.parent.parent for p in INPUTS.rglob('checkpoint.pth')
                         if p.parent.name == 'phase2'
                         and (p.parent.parent / 'phase1/checkpoint_best_mIoU.pth').is_file()
                         and (p.parent.parent / 'pseudo').is_dir()})
    if len(candidates) == 1:
        RESUME_FROM = str(candidates[0])
        print('Recover latest active Phase-2 artifacts from:', RESUME_FROM)
    elif len(candidates) > 1:
        raise RuntimeError('Multiple Phase-2 runs found; set RESUME_FROM to the desired active artifact folder.')
    else:
        raise RuntimeError('Attach the cancelled run active artifact folder containing Phase-2 checkpoint and pseudo-labels. The older Phase-1 ZIP is not enough.')

python = SCRATCH / 'weaktr-env/bin/python'
reuse_environment = False
if python.is_file():
    check_env = os.environ.copy()
    check_env['CUDA_VISIBLE_DEVICES'] = ''
    check_env['PYTHONPATH'] = os.pathsep.join([str(REPO), str(REPO / 'OnlineRetraining')])
    try:
        probe = subprocess.run(
            [str(python), '-c', 'import sys; assert sys.version_info[:2] == (3, 10); import main; import segm.train; import mmcv.ops; import pydensecrf.densecrf'],
            cwd=REPO, env=check_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        reuse_environment = probe.returncode == 0
    except OSError:
        print('Existing interpreter is unusable; setup will repair it.')
print('Reuse installed environment:', reuse_environment)
workflow('prepare', skip_install=reuse_environment)

## 3. Check the completed Phase-1 snapshots

The default resume route requires the full 60-epoch latest snapshot and the best CAM snapshot. This check prints their recorded epochs and score; it fails rather than silently starting new training if the saved run is missing/incomplete. Uploaded files were hash-checked during restoration. Only use checkpoints from your trusted saved run.

In [ ]:
if START_FROM in ('phase2', 'evaluate') and not (WORK / 'weaktr-artifacts/phase2/checkpoint.pth').is_file():
    raise RuntimeError('This resume route needs the actual Phase-2 checkpoint. Recover it from the cancelled run; an earlier Phase-1 ZIP is insufficient.')

if START_FROM != 'phase1':
    check_script = r"""
import json
from pathlib import Path
from reproduction.checkpoint import load_checkpoint
root = Path('/kaggle/working/weaktr-artifacts/phase1')
latest_path = root / 'checkpoint.pth'
best_path = root / 'checkpoint_best_mIoU.pth'
if not latest_path.is_file() or not best_path.is_file():
    raise RuntimeError('Attach the extracted saved run with both Phase-1 checkpoints, then rerun setup.')
latest = load_checkpoint(latest_path)
if latest.get('stage') != 'phase1' or latest.get('format_version') != 1 or latest.get('epoch', -1) < 59:
    raise RuntimeError('The latest Phase-1 snapshot does not confirm 60 completed epochs. Restore the correct run.')
del latest
best = load_checkpoint(best_path)
if best.get('stage') != 'phase1' or best.get('format_version') != 1:
    raise RuntimeError('The best CAM checkpoint is not a full Phase-1 snapshot.')
print(json.dumps({'phase1_completed_epochs': 60, 'best_cam_epoch': best['epoch'] + 1,
                  'best_selection_cam_miou': best.get('miou', best['best']['max_max_mIoU']),
                  'checkpoint': str(best_path)}, indent=2))
"""
    subprocess.run([str(SCRATCH / 'weaktr-env/bin/python'), '-c', check_script], cwd=REPO, check=True)
else:
    print('Phase-1 training route selected explicitly.')
if START_FROM in ('phase2', 'evaluate'):
    phase2_complete()  # Prints the actual restored completed-epoch count before training.


## 4. Optional GPU smoke check

Both model paths were checked in your earlier session. Set `RUN_SMOKE = True` to repeat the synthetic checks after a GPU/environment change. This is separate from a real training epoch and from controlled recovery equivalence.

In [ ]:
if RUN_SMOKE:
    workflow('smoke')
else:
    print('Optional smoke check skipped.')

## 5. Phase 1 — skipped by default

With `START_FROM = 'phase2'` (or `'cams'`), this cell performs no training. Only explicitly selecting `START_FROM = 'phase1'` runs/resumes the official 60-epoch Phase-1 schedule. Your supplied run has already completed it.

In [ ]:
if should_run('phase1'):
    workflow('phase1')
else:
    print('Phase 1 already complete; continuing from the saved best CAM checkpoint.')

## 6. Generate multiscale CAMs

Uses `checkpoint_best_mIoU.pth`, scales 1.0/1.2, and all 10,582 training images. This does not train Phase 1. Complete saved pseudo-labels are reused when available. CAMs are temporary: if a session ends before pseudo-label generation, rerun this stage next session.

In [ ]:
if should_run('cams'):
    workflow('cams')
else:
    print('CAM generation skipped by START_FROM.')

## 7. Evaluate multiscale CAMs

Evaluates the official 1,464-image training selection list across background thresholds 0.40–0.59. Inspect **Best background score ... mIoU ...** before proceeding interactively. The small result file is `/kaggle/working/weaktr-artifacts/cam-eval.txt`. This is a CAM metric, separate from final segmentation validation mIoU. No score threshold blocks the full workflow automatically.

In [ ]:
if should_run('cams'):
    workflow('cam-eval')

## 8. Generate and preserve CRF pseudo-labels

Uses the official fixed threshold argument 43 (0.43), not the threshold selected above. Validates the complete pseudo-label set, then exports the updated resume ZIP. Download it before ending the session. These masks become Phase-2 training targets; author masks are not substituted.

In [ ]:
if should_run('pseudo'):
    workflow('pseudo')
    workflow('preserve')
else:
    print('Using previously saved pseudo-labels; Phase 2 validates their complete coverage.')

## 9. Finish Phase 2: 89 -> 100 epochs

The guard above verifies that at least 89 epochs are already saved. This cell trains at most 11 additional epochs, clamped to the original 100-epoch total, preserving SGD/poly scheduling, total batch 4, resize 520/crop 480, gradient clipping 1.2/120 and validation every epoch. A 100-epoch checkpoint is saved and exported before final evaluation begins.

Keep `PHASE2_PILOT=False` for this final run. The historical pilot option is only for deliberately performing a short check; it skips final evaluation. The earlier Phase-1/CAM cells are skipped automatically by START_FROM.

In [ ]:
if should_run('phase2'):
    workflow('phase2-pilot' if PHASE2_PILOT else 'phase2')
    if PHASE2_PILOT:
        print('Phase-2 pilot complete. Download weaktr-resume.zip. Set PHASE2_PILOT=False and START_FROM=phase2 to continue.')

## 10. Final multiscale evaluation, CRF, and report

Runs only after the checkpoint confirms 100 completed epochs. Preserves official two-decoder inference (`backbone=multi`, weight 0.8), then CRF postprocessing on all 1,449 validation images.

Small result files in `weaktr-artifacts/`:

- `phase2/scores_ms.yml`: raw multiscale mean/per-class scores.
- `phase2/scores_ms_crf.json`: post-CRF mean/per-class scores and validation count.
- `final-summary.json` and `final-report.txt`: both scores, the gap to the 74.0% reference, checkpoint hash, and stage runtimes.

The report requires completed evaluation records tied to the actual final checkpoint and complete CRF prediction coverage. It refuses incomplete or stale results. The final ZIP contains reports/scores/checkpoints/configs/logs/pseudo-labels. Final prediction masks remain in `weaktr-artifacts/seg_pred_ms/` in notebook outputs; temporary probabilities are excluded from the ZIP.

In [ ]:
if should_run('evaluate') and not PHASE2_PILOT:
    if phase2_complete():
        try:
            workflow('evaluate')
            workflow('crf')
            workflow('report')
        finally:
            workflow('preserve')
    else:
        raise RuntimeError('Final-session training did not reach epoch 100. Preserve the checkpoint and resume before final evaluation.')
else:
    print('Final evaluation skipped during the Phase-2 pilot.')

## Download final results

When all stages succeed, download `/kaggle/working/weaktr-resume.zip` and `weaktr-artifacts/final-summary.json` (or `final-report.txt`). The ZIP contains the small final score files/reports and full training recovery state. Preserve saved notebook outputs to retain the final PNG prediction collection too.

If evaluation fails after training finishes, the 100-epoch checkpoint ZIP has already been exported. In a fresh session attach that saved run, set `START_FROM='evaluate'`, run setup/verification, then the final evaluation cell. This regenerates temporary probabilities and CRF outputs without retraining or regenerating training pseudo-labels.

The final summary establishes that the training/evaluation run completed; matching the paper and explaining any remaining gap requires reviewing the measured scores. Keep the original inputs and all saved-run provenance.

In [ ]:
if PHASE2_PILOT:
    print('Pilot only; final results are not claimed. Download the checkpoint ZIP to continue.')
else:
    summary_path = WORK / 'weaktr-artifacts/final-summary.json'
    if not summary_path.is_file():
        raise RuntimeError('Final summary is missing; inspect the failed stage before declaring this run complete.')
    summary = json.loads(summary_path.read_text())
    print('TRAINING AND FINAL EVALUATION COMPLETED')
    print('Raw multiscale mIoU:', summary['raw_multiscale_miou'])
    print('Post-CRF mIoU:', summary['post_crf_miou'])
    print('Shortfall to paper target (percentage points):', summary['paper_target_shortfall_pp'])
    print('Download:', WORK / 'weaktr-resume.zip')
    print('Final report:', summary_path)